In [8]:
import numpy as np
from PIL import Image

def load_and_preprocess_image(image_path, image_size=(256, 256), grayscale=True):
    img = Image.open(image_path)
    if grayscale:
        img = img.convert('L')  # 흑백 변환
    else:
        img = img.convert('RGB')  # RGB 변환
    img = img.resize(image_size)
    img_array = np.array(img) / 255.0  # 정규화

    if grayscale:
        img_array = img_array.reshape(1, image_size[0], image_size[1])  # (1, H, W)
    else:
        img_array = img_array.transpose(2, 0, 1)  # (C, H, W)

    return img_array

# 예시 사용
# image_path = 'path_to_your_image.jpg'
# X = load_and_preprocess_image(image_path)
# print("Preprocessed Image Shape:", X.shape)  # (1, 256, 256) 또는 (3, 256, 256)


In [9]:
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    Parameters:
        input_data : Input data of shape (N, C, H, W)
        filter_h : Height of the filter
        filter_w : Width of the filter
        stride : Stride size
        pad : Padding size

    Returns:
        col : 2D array of shape (N*out_h*out_w, C*filter_h*filter_w)
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1

    img = np.pad(input_data, 
                 [(0,0), (0,0), (pad, pad), (pad, pad)], 
                 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col


In [10]:
def convolution_im2col(input_data, filters, stride=1, pad=0):
    """
    Perform convolution using im2col.

    Parameters:
        input_data : Input data of shape (N, C, H, W)
        filters : Filter weights of shape (F, C, HH, WW)
        stride : Stride size
        pad : Padding size

    Returns:
        out : Output data after convolution, shape (N, F, out_h, out_w)
    """
    N, C, H, W = input_data.shape
    F, _, HH, WW = filters.shape

    # Calculate output dimensions
    out_h = (H + 2*pad - HH) // stride + 1
    out_w = (W + 2*pad - WW) // stride + 1

    # Transform input and filters
    col = im2col(input_data, HH, WW, stride, pad)  # Shape: (N*out_h*out_w, C*HH*WW)
    reshaped_filters = filters.reshape(F, -1).T  # Shape: (C*HH*WW, F)

    # Perform matrix multiplication
    out = np.dot(col, reshaped_filters)  # Shape: (N*out_h*out_w, F)

    # Reshape output to (N, F, out_h, out_w)
    out = out.reshape(N, out_h, out_w, F).transpose(0, 3, 1, 2)
    return out


In [11]:
import numpy as np
if __name__ == "__main__":
    # 입력 데이터 (N, C, H, W)
    input_data = np.random.randn(1, 1, 256, 256)  # 흑백 이미지

    # 필터 (F, C, HH, WW)
    filters = np.random.randn(32, 1, 3, 3)  # 32개의 3x3 필터

    # 스트라이드와 패딩 설정
    stride = 1
    pad = 1

    # 합성곱 수행
    output = convolution_im2col(input_data, filters, stride, pad)

    print("Input shape:", input_data.shape)        # (1, 1, 256, 256)
    print("Filters shape:", filters.shape)        # (32, 1, 3, 3)
    print("Output shape:", output.shape)          # (1, 32, 256, 256)
    print("Output data:\n", output)


Input shape: (1, 1, 256, 256)
Filters shape: (32, 1, 3, 3)
Output shape: (1, 32, 256, 256)
Output data:
 [[[[ -1.01386259   0.86501435  -3.78655985 ...  -0.77636473
     -0.21223402   0.40081965]
   [ -0.64380713   3.06601066   2.10267804 ...   1.73609874
     -0.82918882  -1.09913169]
   [  4.29403113  -0.11785224  -2.11952448 ...   1.58766955
      0.08821508  -0.75536257]
   ...
   [  2.31159941  -4.28114195   1.91098205 ...   3.57843176
     -4.51176839   5.10607758]
   [ -2.83550931   4.46319974   5.54465222 ...  -4.19099273
      0.62818999   2.78102468]
   [ -2.3085936   -5.8738478   -0.74660179 ...  -1.45114119
     -3.57609113   2.96127849]]

  [[ -5.876444    -1.93165078   0.32576231 ...   1.88036891
     -0.25277369   1.01103466]
   [ -3.17494648  -1.88923723   4.16579454 ...  -1.21377269
     -0.20596986  -1.37702108]
   [  0.04470898  -3.74307375   3.15687993 ...   0.59242892
      5.11124673   0.90884717]
   ...
   [  3.2654487    5.95620706   0.95695212 ...  -0.11860632
